In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch.utils.data as data
from scipy.stats import shapiro
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import os

In [7]:
# Load the data
data = pd.read_csv('inputFGclassifierWithLabel.CSV')

# Preprocess the data
# Drop the first column (index 0) and the 13th column (index 12)
data_processed = data.drop(data.columns[[0, 12]], axis=1)

# Select columns 2 to 19 (index 1 to 18 in Python, as index starts from 0)
features = data_processed.iloc[:, 0:17]

# Get the last 10 columns as labels
labels = data_processed.iloc[:, -10:]

# Display processed data information
print(f"Processed data shape:")
print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Labels columns: {labels.columns.tolist()}")

print(f"\nFeature statistics:")
print(features.describe())

Processed data shape:
Features shape: (24528, 17)
Labels shape: (24528, 10)
Labels columns: ['LabelFillingP', 'LabelEF', 'LabelCO', 'LabelLAV', 'LabelRVO2E', 'LabelLVO2E', 'LabelLVMVO2', 'LabelRVMVO2', 'LabelLVME', 'LabelRVME']

Feature statistics:
                Sex        Height        Weight           SBP           DBP  \
count  24528.000000  24528.000000  24528.000000  24528.000000  24528.000000   
mean       0.469300    171.587851     91.603987    126.032127     74.564579   
std        0.499067      8.636087     23.191223     24.965296     16.007327   
min        0.000000    138.300000     40.400000     47.000000     29.000000   
25%        0.000000    165.500000     75.600000    109.000000     64.000000   
50%        0.000000    171.300000     88.000000    124.000000     74.000000   
75%        1.000000    177.600000    103.300000    142.000000     84.000000   
max        1.000000    207.500000    337.800000    236.000000    150.000000   

                 EF       Hed_SW       

In [10]:

# Normalize the input features
# Skip the first and last feature columns (binary variables)
continuous_features = features.iloc[:, 1:-1]
scaler = MinMaxScaler()
normalized_features = scaler.fit_transform(continuous_features)

# Combine the binary and normalized continuous features
normalized_features = pd.DataFrame(normalized_features, columns=continuous_features.columns)
normalized_features.insert(0, features.columns[0], features.iloc[:, 0])  # Add the first binary column
normalized_features[features.columns[-1]] = features.iloc[:, -1]  # Add the last binary column

# Convert normalized features to PyTorch Tensor
input_tensor = torch.tensor(normalized_features.values, dtype=torch.float32)

# Display the tensor and its size
print("Input tensor:")
print(input_tensor)
print("\nTensor size:", input_tensor.size())

Input tensor:
tensor([[1.0000, 0.4379, 0.1516,  ..., 0.1392, 0.2769, 0.0000],
        [0.0000, 0.5116, 0.1163,  ..., 0.1418, 0.3385, 1.0000],
        [0.0000, 0.6358, 0.2455,  ..., 0.1873, 0.1308, 0.0000],
        ...,
        [1.0000, 0.5520, 0.3376,  ..., 0.1823, 0.1923, 0.0000],
        [1.0000, 0.6243, 0.4593,  ..., 0.1392, 0.2385, 0.0000],
        [0.0000, 0.4942, 0.1715,  ..., 0.2380, 0.2385, 1.0000]])

Tensor size: torch.Size([24528, 17])


In [11]:
# Define the MLP model
class MLPClassifier(nn.Module):
    def __init__(self, input_size):
        super(MLPClassifier, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),  # First hidden layer
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.5),
            nn.Linear(64, 32),  # Second hidden layer
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.5),
            nn.Linear(32, 1),  # Output layer
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

# Initialize the model
input_size = 17  # Number of input features
model = MLPClassifier(input_size)

# Define the loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Display the model structure
print(model)

MLPClassifier(
  (model): Sequential(
    (0): Linear(in_features=17, out_features=64, bias=True)
    (1): ReLU()
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): Dropout(p=0.5, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)


In [28]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    input_tensor.numpy(), labels.values, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

# Training parameters
batch_size = 4
num_epochs = int(1e4) 
learning_rate = 0.0001

# DataLoader for batching
train_dataset = data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Directory to save models
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)


Using device: cpu


In [27]:
# Loop through each label column
for i in range(labels.shape[1]):
    label_name = labels.columns[i]  # Use label name for model identification
    print(f"Training classifier for label {label_name}...")
    
    # Initialize model, loss function, and optimizer
    model = MLPClassifier(input_size=17).to(device)  # Move model to device
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    best_val_auc = 0.0  # Track the best validation AUC
    early_stop = False  # Early stopping flag

    # Training loop
    for epoch in range(num_epochs):
        if early_stop:
            break

        model.train()
        epoch_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            # Move data to device
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Select the current label column
            batch_y = batch_y[:, i].unsqueeze(1)  # Shape: (batch_size, 1)
            
            # Forward pass
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        # Validation
        model.eval()
        with torch.no_grad():
            X_val_tensor_device = X_val_tensor.to(device)  # Move validation data to device
            val_outputs = model(X_val_tensor_device)
            val_labels = y_val_tensor[:, i].unsqueeze(1).to(device)
            val_auc = roc_auc_score(val_labels.cpu().numpy(), val_outputs.cpu().numpy())

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Val AUC: {val_auc:.4f}")

        # Check for early stopping
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            # Save the best model
            model_path = os.path.join(save_dir, f"model_{label_name}_best.pth")
            torch.save(model.state_dict(), model_path)
            print(f"New best model for label {label_name} saved with AUC: {best_val_auc:.4f}")

        if best_val_auc >= 0.975:
            print(f"Early stopping triggered for label {label_name} at epoch {epoch+1} with AUC: {best_val_auc:.4f}")
            early_stop = True

    print(f"Finished training classifier for label {label_name}.")

Training classifier for label LabelFillingP...
Epoch 1/20000, Loss: 3451.4745, Val AUC: 0.6973
New best model for label LabelFillingP saved with AUC: 0.6973
Epoch 2/20000, Loss: 3282.2363, Val AUC: 0.7364
New best model for label LabelFillingP saved with AUC: 0.7364
Epoch 3/20000, Loss: 3172.4041, Val AUC: 0.7554
New best model for label LabelFillingP saved with AUC: 0.7554


KeyboardInterrupt: 